# Flight Delay Training Pipeline Walkthrough

This notebook shows the training pipeline step by step. It is meant for team members who want to understand what happens without reading all Python modules first.

Goal: classify the departure delay into four classes two hours before departure:

- `no_delay`: delay <= 15 minutes
- `small_delay`: 15 minutes < delay <= 1 hour
- `medium_delay`: 1 hour < delay <= 3 hours
- `large_delay`: delay > 3 hours


## 1. Setup

Run this notebook from the `pipeline/` folder.

In Colab, mount Drive and set `DATA_ROOT` below. If imports fail in Colab, install the needed packages once:

```python
%pip install pandas numpy scikit-learn pyarrow s3fs joblib
```


In [1]:
from pathlib import Path
import os
import sys
import importlib
import pandas as pd
from dotenv import load_dotenv

# Make local imports work when the notebook is opened from another folder.
PIPELINE_DIR = Path.cwd()
if not (PIPELINE_DIR / 'loader.py').exists():
    PIPELINE_DIR = Path.cwd() / 'pipeline'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.append(str(PIPELINE_DIR))

from loader import LoaderStorage
from targets import add_delay_class_target, DELAY_CLASS_ORDER
from split import chronological_train_val_test_split
from features import build_feature_matrix, fit_transform, transform, check_columns
from evaluate import evaluate_classifier, classification_report_frame, f1_macro_score, plotConfusionMatrix
from train import TrainingConfig, run_training
import models
importlib.reload(models)
from models import make_model,make_neural_network_model, get_feature_importances, plot_feature_importances


In [2]:
from models import make_model,make_neural_network_model, get_feature_importances, plot_feature_importances


## 2. Configuration

Only change this cell for normal usage.

Examples:

- Colab/Drive: `DATA_ROOT = "/content/drive/MyDrive/Datamining"`
- S3: `DATA_ROOT = "s3://data-mining"`
- Local folder: `DATA_ROOT = "."`


In [3]:
# Change these paths to your actual dataset location.
DATA_ROOT = "s3://data-mining"
INPUT_PATH = 'data/features/feature_engineered.parquet'

# Column names used by the current pipeline.
DELAY_COLUMN = 'ArrDelayMinutes'
TIME_COLUMN = 'CRSDepDateTime_UTC'
TARGET_COLUMN = 'delay_class'

# Use a small sample while learning/debugging. Set to 1.0 for the final run.
SAMPLE_FRAC = 0.05

# Good first choices: 'dummy', 'logistic_regression', 'random_forest', 'hist_gradient_boosting', xgboost.
MODEL_NAME = 'xgboost'

OUTPUT_DIR = 'outputs/training_notebook'


## 3. Load The Data

`LoaderStorage` hides whether the file is loaded from local disk, Google Drive, or S3. The rest of the notebook can use the same code for all three.


In [4]:
storage = LoaderStorage(DATA_ROOT)

if INPUT_PATH.endswith('.parquet'):
    df = storage.read_parquet(INPUT_PATH)
elif INPUT_PATH.endswith('.csv'):
    df = storage.read_csv(INPUT_PATH)
else:
    raise ValueError('Use a .parquet or .csv dataset.')

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=42)

print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns):,}')
df.head()


Rows: 93,369
Columns: 96


,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,Reporting_Airline,Origin,Dest,ArrDelayMinutes,CRSElapsedTime,...,hist_flg_median_delay_30d,hist_flg_count_delay_30d,origin_yesterday_delay,origin_lastweek_delay,dest_yesterday_delay,dest_lastweek_delay,airline_yesterday_delay,airline_lastweek_delay,global_yesterday_delay,global_lastweek_delay
1053466,347259,2014,6,7,6,UA,DCA,IAH,20.0,187.0,...,3.0,30.0,11.252336,1.141176,13.610465,10.923077,14.022727,7.296791,13.154360,6.552715
649126,106842,2014,8,5,2,DL,FLL,ATL,0.0,117.0,...,0.0,30.0,50.336283,15.633929,14.367292,5.347107,11.620991,4.696108,18.977261,11.514601
1680866,347563,2014,7,21,1,WN,BWI,AUS,36.0,200.0,...,7.0,30.0,9.753968,29.417323,10.453608,37.880000,12.848271,22.975342,8.078544,27.283015
57031,18704,2014,5,15,4,AA,ORD,SEA,23.0,260.0,...,0.0,30.0,17.106007,23.775801,9.625698,17.887006,12.345309,57.600246,15.231400,28.883910
1370247,445759,2014,5,23,5,UA,LAX,JFK,36.0,337.0,...,0.0,30.0,20.824503,14.967532,53.919463,65.929032,27.093719,23.634766,26.547145,21.731089


## 4. Create The Target Classes

The raw delay in minutes is converted into the four interval classes. After this step, the model predicts `delay_class`, not the exact number of minutes.


In [5]:
df = add_delay_class_target(
    df,
    delay_column=DELAY_COLUMN,
    target_column=TARGET_COLUMN,
)

# Show the class balance. This is important because large delays are usually rare.
class_distribution = (
    df[TARGET_COLUMN]
    .value_counts()
    .reindex(DELAY_CLASS_ORDER, fill_value=0)
    .to_frame(name='count')
)

class_distribution['share'] = class_distribution['count'] / len(df)
# expected amount in the test set (20% of the data)
class_distribution['expected_in_test'] = class_distribution['count'] * 0.15
display(class_distribution)

,count,share,expected_in_test
delay_class,,,
no_delay,73299,0.785046,10994.85
small_delay,14207,0.152160,2131.05
medium_delay,5070,0.054301,760.50
large_delay,793,0.008493,118.95


In [6]:
# display all the records, that are in the Large delay class
display(df[df["delay_class"] == 'large_delay'])

,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,Reporting_Airline,Origin,Dest,ArrDelayMinutes,CRSElapsedTime,...,hist_flg_count_delay_30d,origin_yesterday_delay,origin_lastweek_delay,dest_yesterday_delay,dest_lastweek_delay,airline_yesterday_delay,airline_lastweek_delay,global_yesterday_delay,global_lastweek_delay,delay_class
328856,1797,2014,8,17,7,AA,DFW,SAN,192.0,170.0,...,30.0,27.023622,29.445614,6.776923,16.712329,17.232877,19.972222,8.779441,11.802652,large_delay
905181,117457,2014,6,18,3,DL,ORD,ATL,237.0,115.0,...,30.0,9.204082,25.971429,11.457300,22.859504,8.540824,22.939006,10.758916,25.146974,large_delay
474644,410477,2014,1,5,7,B6,JFK,LAX,329.0,374.0,...,2.0,142.572650,0.000000,43.339768,0.000000,157.264317,0.000000,48.868026,0.000000,large_delay
536872,69933,2014,8,20,3,B6,SFO,JFK,202.0,322.0,...,30.0,19.995868,8.849421,19.942857,14.611429,12.973373,31.537538,20.064774,13.385845,large_delay
1671591,368806,2014,7,26,6,WN,HOU,DAL,424.0,60.0,...,19.0,11.557522,16.126437,8.311111,16.625000,18.904155,14.228659,11.157941,8.187748,large_delay
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754480,182191,2014,6,11,3,WN,MCO,AUS,228.0,155.0,...,6.0,38.122807,6.076923,18.447619,8.724490,28.987179,10.867647,21.181089,8.830761,large_delay
660079,120460,2014,6,19,4,DL,ORD,MSP,318.0,88.0,...,30.0,101.063745,25.978495,31.065789,13.294118,23.822521,13.950186,30.219298,27.396240,large_delay
1700201,295771,2014,7,3,4,WN,PHX,LAS,210.0,65.0,...,30.0,21.098684,19.980892,17.467593,15.526786,26.477936,20.254654,27.856851,18.437999,large_delay
180030,32975,2014,6,25,3,AA,DFW,AUS,202.0,50.0,...,30.0,42.316498,28.191781,28.851485,23.050505,26.757101,38.871327,20.158565,30.219298,large_delay


In [7]:
# check if all expected Columsn are present, and what unexpected ones we have.
# drop the Unnamed:0 column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
check_columns(df)


Info: The following columns are in the dataset but not recorded in the feature engineering lists: {'full_flight_number', 'CRSDepDateTime_UTC'}


## 5. Chronological Train / Validation / Test Split

For a forecasting-like task, we should not randomly mix old and future flights. The model trains on older flights and is evaluated on later flights.

- Train: first 70% of time
- Validation: next 15%
- Test: final 15%


In [8]:
train_df, val_df, test_df = chronological_train_val_test_split(
    df,
    time_column=TIME_COLUMN,
)

print(f'Train rows:      {len(train_df):,}')
print(f'Validation rows: {len(val_df):,}')
print(f'Test rows:       {len(test_df):,}')

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'start': [train_df[TIME_COLUMN].min(), val_df[TIME_COLUMN].min(), test_df[TIME_COLUMN].min()],
    'end': [train_df[TIME_COLUMN].max(), val_df[TIME_COLUMN].max(), test_df[TIME_COLUMN].max()],
    'rows': [len(train_df), len(val_df), len(test_df)],
})
# drop the time column to avoid data leakage. We will use it later to check the chronological order of the splits.
train_df = train_df.drop(columns=[TIME_COLUMN])
val_df = val_df.drop(columns=[TIME_COLUMN])
test_df = test_df.drop(columns=[TIME_COLUMN])

Train rows:      65,358
Validation rows: 14,005
Test rows:       14,006


## 6. Build Features

This step separates `X` and `y`:

- `X`: the input columns the model is allowed to use
- `y`: the delay class the model should learn to predict

The helper also drops obvious leakage columns like actual delay, actual departure time, actual arrival time, etc.


In [9]:
X_train, y_train = fit_transform(train_df, scale="minmax") # alternatively "standard"
X_val, y_val = transform(val_df)
X_test, y_test = transform(test_df)



print(f'Number of model features: {len(X_train.columns):,}')
X_train.head()
# plot a cross correlation matrix of the features

# for every picture, save a violin plot of the feature distribution for each class


Info: The following columns are in the dataset but not recorded in the feature engineering lists: {'full_flight_number'}


KeyError: "['route_delay'] not found in axis"

In [ ]:
y_train.head()

## 7. Train A Simple Baseline

Always train a baseline first. If a complex model does not beat this, something is wrong or the features are weak.


In [ ]:
baseline = make_model('dummy')
baseline.fit(X_train, y_train)

baseline_metrics, baseline_predictions = evaluate_classifier(baseline, X_test, y_test)
display(pd.Series(baseline_metrics, name='baseline_validation_metrics'))

class_distribution = (
    y_train
    .value_counts()
    .to_frame(name='count')
)
display(class_distribution)

In [ ]:
baseline = make_model('Baseline')
baseline.fit(X_train, y_train)

baseline_metrics, baseline_predictions = evaluate_classifier(baseline, X_test, y_test)
pd.Series(baseline_metrics, name='baseline_validation_metrics')


## 8. Train The Selected Model

Now train the model chosen in the configuration cell. For large datasets, start with a small sample and only later set `SAMPLE_FRAC = 1.0`.


In [ ]:
import inspect
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
import numpy as np




class_weight_dict = {0: 0.3,
                     1: 0.7,
                     2: 1.1,
                     3: 1.5}
sample_weights = compute_sample_weight(class_weight=class_weight_dict, y=y_train)

model = make_model(MODEL_NAME)
# fit_model_with_class_weights(model, X_train, y_train)

model.fit(X_train, y_train,sample_weight=sample_weights)
val_metrics, val_predictions = evaluate_classifier(model, X_test, y_test)
pd.Series(val_metrics, name=f'{MODEL_NAME}_validation_metrics')


In [ ]:
# model = make_neural_network_model(input_shape=X_train.shape[1], num_classes=len(DELAY_CLASS_ORDER))
# # fit_model_with_class_weights(model, X_train, y_train)
# hist =model.fit(X_train, y_train, epochs=5, batch_size=256, validation_data=(X_val, y_val),sample_weight=sample_weights,shuffle=True)
# f1_macro = f1_macro_score(model, X_val, y_val)
# print(f"Validation f1-macro score: {f1_macro:.4f}")
# val_metrics, neuralNetwork_predictions = evaluate_classifier(model, X_test, y_test)
# pd.Series(val_metrics, name=f'NN_validation_metrics')

# # plot the training history
# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# hist_df = pd.DataFrame(hist.history)
# axes[0].plot(hist.history["loss"], label="Training Loss")
# axes[0].plot(hist.history["val_loss"], label="Validation Loss")
# axes[0].set_xlabel("Epoch")
# axes[0].set_ylabel("Loss")
# axes[0].legend()



# axes[1].plot(hist.history["f1_score"], label="Training F1-Score")
# axes[1].plot(hist.history["val_f1_score"], label="Validation F1-Score")
# axes[1].set_xlabel("Epoch")
# axes[1].set_ylabel("F1-Score")
# axes[1].legend()

# axes[2].plot(hist.history["accuracy"], label="Training Accuracy")
# axes[2].plot(hist.history["val_accuracy"], label="Validation Accuracy")
# axes[2].set_xlabel("Epoch")
# axes[2].set_ylabel("Accuracy")
# axes[2].legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# plot baseline confusion matrix
plotConfusionMatrix(predictions=baseline_predictions, y_true=y_test)
plotConfusionMatrix(predictions=val_predictions, y_true=y_test)


In [ ]:
# get the feature importances for the model and plot them
feature_importances = get_feature_importances(model, feature_names=X_train.columns)
# plot the feature importances
plot_feature_importances(feature_importances, top_n=30)
display(feature_importances)

In [ ]:
# without class weights
model2 = make_model(MODEL_NAME)
# fit_model_with_class_weights(model, X_train, y_train)
model2.fit(X_train, y_train)
val_metrics, val_predictions = evaluate_classifier(model2, X_test, y_test)
display(pd.Series(val_metrics, name=f'{MODEL_NAME}_validation_metrics'))
plotConfusionMatrix(predictions=val_predictions, y_true=y_test)
# get the feature importances for the model and plot them
feature_importances = get_feature_importances(model2, feature_names=X_train.columns)
# plot the feature importances
plot_feature_importances(feature_importances, top_n=25)

In [ ]:
test_metrics = evaluate_classifier(model, X_test, y_test)
pd.Series(test_metrics, name=f'{MODEL_NAME}_test_metrics')

## 9. Final Test Evaluation

Only use the test set after choosing a model. This gives the honest final estimate for the report.


In [ ]:
test_metrics = evaluate_classifier(model, X_test, y_test)
pd.Series(test_metrics, name=f'{MODEL_NAME}_test_metrics')


In [ ]:
# Per-class report. Look especially at recall for medium_delay and large_delay.
classification_report_frame(model, X_test, y_test)
